|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Chunked prefill<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: one token budget, two kinds of work<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the mixed-batch scheduler.

Each step has one token budget. A sequence that decodes contributes one token.
A sequence that prefills contributes a slice of its prompt. The scheduler
never lets a step become a long prefill that stalls everybody.

This is stage 11. It is the last scheduler in the course that is pure logic.

In [ ]:
### run this cell

# 32 short conversations that arrive at once and settle into decoding,
# then at step 40 somebody pastes in 4096 tokens.
#   [arrival_step, prompt_len, output_len]
reqs = [[0, 64, 400] for _ in range(32)] + [[40, 4096, 100]]
print(f'{len(reqs)} requests; the big one arrives at step {reqs[-1][0]}')

# Exercise 1: the budget loop

Every step spends a fixed number of tokens. Decodes first, then as much of
the waiting prompts as still fits.

In [ ]:
def schedule(requests, budget, max_running=64):
  """requests: [[arrival_step, prompt_len, output_len], ...].
  Returns an array of (tokens_in_step, sequences_decoding_that_step).

  One budget per step. A decoding sequence contributes 1 token. A
  prefilling one contributes as much of its prompt as still fits."""
  pending = sorted(requests)
  waiting, prefilling, decoding = [], [], []
  steps, t = [], 0

  while pending or waiting or prefilling or decoding:
    while pending and pending[0][0] <= t:
      _, p, o = pending.pop(0); waiting.append([p, o])
    while waiting and len(prefilling) + len(decoding) < max_running:
      prefilling.append(waiting.pop(0))

    used = 0
    n_decoding = len(decoding)      # how many users are waiting on this step
    # decodes first. Why first? They are one token each, and a user
    # waiting on a continuation notices sooner than one waiting to start.
    for d in list(decoding):
      

    # then spend the remainder of the budget on prompt chunks
    for pr in list(prefilling):
      
      # a prompt that finishes becomes a decoding sequence
      

    steps.append((used, n_decoding)); t += 1
    if used == 0 and not pending: break
  return np.array(steps)

s = schedule(reqs, budget=512)
print(f'{len(s)} steps, mean {s[:,0].mean():.0f} tokens, max {s[:,0].max()} tokens')

# Exercise 2: put a cost on it, in the correct units

A step is not proportional to the tokens in it. The notebook
`part4_chk_theStall` measured the shape. The cost is flat up to a few hundred
tokens and linear after that. The position of the bend is a property of your
card.

So do not write milliseconds. Write the cost in **units of one plain decode
step**, and make the bend a parameter. Every number below is then a ratio, and
Exercise 3 can change the machine and change nothing else.

In [ ]:
# Step cost is not a table of milliseconds. It is Part 1's roofline again:
# a step is free up to KNEE tokens and linear after that.
#
# KNEE is the only hardware number in this notebook. Measure yours in
# part4_chk_theStall: the largest token budget whose step still costs what a
# 32-token step costs.
KNEE = 256

def step_cost(n):
  """Cost of a step holding n tokens, in units of one plain decode step."""
  return 

print(f"{'budget':>7} {'steps':>7} {'total cost':>11} {'p99 ITL':>9} {'worst ITL':>11}")
for budget in (64, 128, 256, 512, 1024, 4096):
  s   = schedule(reqs, budget)
  c   = 
  # a p99 over STEPS is the wrong population: one catastrophic step in two
  # hundred does not reach p99, and every user felt it. Weight each step by
  # how many sequences were decoding through it.
  itl = 
  print(f'{budget:>7} {len(s):>7} {c.sum():>10.0f}x {np.percentile(itl,99):>8.2f}x '
        f'{itl.max():>10.2f}x')
print('\nall costs are multiples of one plain decode step')

# Exercise 3: change the machine

Everything above used one `KNEE`. Sweep it and find the best budget for each
machine. You are looking for a rule relating the two, not three numbers.

In [ ]:
budgets = [64, 128, 256, 512, 1024, 2048, 4096]

# Sweep the machine, not the workload. For each KNEE, which budget is best?
print(f"{'KNEE':>6} " + ' '.join(f'{b:>7}' for b in budgets))
for knee in (64, 256, 1024):
  row = []
  for b in budgets:
    
  best = budgets[int(np.argmin(row))]
  print(f'{knee:>6} ' + ' '.join(f'{v:>7.0f}' for v in row) + f'   best budget {best}')

### Before you open the solution

1. In Exercise 3, how does the best budget relate to `KNEE`? Write the rule
   down.
2. Your p99 ITL stops improving below a certain budget. Why does it flatten
   there, and which Part 1 measurement predicts that point?
3. **Look at the largest budget.** Its p99 is excellent and its worst ITL is
   terrible. Work out how many steps were catastrophic, how many users sat
   through each, and what fraction of all token-waits that is. Then decide
   which number you would put on a dashboard.
4. You spend the budget on decodes before prefill chunks. What breaks if you
   swap the order, and who notices?